# Travel Universe — iPad Colab reconstruction runner

This notebook runs a **manual, temporary** reconstruction job using Google Colab. It uses the public Travel Universe source code and the original video you select from your iPad. It does not connect to your GitHub account, push files to GitHub, or request GitHub credentials.

## Privacy and safety
- Do not enter passwords, access tokens, API keys, or private account details in this notebook.
- This notebook does **not** mount Google Drive, so its code does not receive broad access to your Drive files. The uploaded video and results live in this temporary Colab runtime until you download the result.
- Only select the Travel Universe test video. Do not upload unrelated private files.
- Review the notebook source before running it. It installs software from the official COLMAP, gsplat, and Niantic SPZ projects.
- Free GPU access is not guaranteed. If Colab does not assign a GPU, stop; do not buy compute for this test.

Run the cells from top to bottom using the play button beside each cell.

## 1. Check the cloud GPU
In Colab, choose **Runtime → Change runtime type → T4 GPU** if available, then run this cell. A GPU is required for the Gaussian Splat training stage.

In [ ]:
import torch, subprocess, sys, platform
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No free GPU was assigned. Stop here and try another time; no paid upgrade is needed.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=False)


## 2. Install the required tools
This installs FFmpeg and COLMAP from the Colab system package source. It also downloads the public Travel Universe, gsplat, and SPZ source repositories.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def run(command, *, env=None, cwd=None):
    print('\n>>>', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, check=True, env=env, cwd=cwd)

# Install system tools and native libraries used by COLMAP, gsplat, and SPZ.
run(['apt-get', 'update', '-qq'])
run(['apt-get', 'install', '-y', 'ffmpeg', 'colmap', 'build-essential', 'cmake', 'ninja-build', 'zlib1g-dev', 'libzstd-dev'])

# Fetch clean public source checkouts. No GitHub login or Drive access is used.
for path in ['/content/Travel-universe', '/content/gsplat', '/content/spz']:
    if os.path.exists(path):
        import shutil
        shutil.rmtree(path)
run(['git', 'clone', '--depth', '1', 'https://github.com/virtualearthworld/Travel-universe.git', '/content/Travel-universe'])
run(['git', 'clone', '--depth', '1', 'https://github.com/nerfstudio-project/gsplat.git', '/content/gsplat'])
run(['git', 'clone', '--depth', '1', 'https://github.com/nianticlabs/spz.git', '/content/spz'])

# Defer gsplat CUDA compilation until runtime; preserve Colab's CUDA PyTorch.
gsplat_env = os.environ.copy()
gsplat_env['BUILD_NO_CUDA'] = '1'
run([sys.executable, '-m', 'pip', 'install', '-e', '/content/gsplat'], env=gsplat_env)

# Install only trainer runtime dependencies; avoid replacing Colab's torch/torchvision.
run([sys.executable, '-m', 'pip', 'install', 'imageio[ffmpeg]', 'tyro', 'viser', 'pyyaml', 'tqdm', 'torchmetrics', 'lpips', 'nerfview @ git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787', 'scipy', 'scikit-learn', 'tensorboard', 'opencv-python-headless', 'Pillow', 'piexif', 'tensorly', 'matplotlib', 'splines'])

# Build SPZ's official C++ PLY-to-SPZ converter. This avoids the separate
# nanobind/scikit-build Python extension build that failed in the previous run.
SPZ_BUILD = '/content/spz/build'
run(['cmake', '-S', '/content/spz', '-B', SPZ_BUILD, '-G', 'Ninja', '-DCMAKE_BUILD_TYPE=Release', '-DSPZ_BUILD_TOOLS=ON', '-DSPZ_BUILD_PYTHON_BINDINGS=OFF', '-DSPZ_BUILD_EXTENSIONS=OFF'])
run(['cmake', '--build', SPZ_BUILD, '--target', 'ply_to_spz', 'spz_info', '-j2'])
SPZ_CONVERTER = Path(SPZ_BUILD) / 'ply_to_spz'
assert SPZ_CONVERTER.is_file(), 'SPZ converter executable was not created.'

# Verify imports and required system tools before continuing.
import torch, gsplat
print('\nGPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('gsplat import: OK')
print('SPZ converter:', SPZ_CONVERTER)
run(['ffmpeg', '-version'])
run(['colmap', '-h'])
print('\nInstallation and verification completed successfully.')


## 3. Select the original Travel Universe video
Use the file picker to select the original MOV video from your iPad. The video is uploaded to this temporary Colab session, not to GitHub.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, os
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Please select exactly one video file and run this cell again.')
video_name = next(iter(uploaded))
if Path(video_name).suffix.lower() not in {'.mov', '.mp4', '.webm'}:
    raise RuntimeError('Please select an MP4, MOV, or WebM video.')
VIDEO = Path('/content') / video_name
print('Selected video:', VIDEO.name, 'size:', round(VIDEO.stat().st_size / 1_000_000, 1), 'MB')
!ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 "$VIDEO"


## 4. Extract frames using the app's sampling settings
The Travel Universe worker samples two frames per second and caps the long image edge at 1600 pixels.

In [ ]:
import subprocess
DATA = Path('/content/tu_dataset')
IMAGES = DATA / 'images'
IMAGES.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'ffmpeg', '-hide_banner', '-loglevel', 'error', '-i', str(VIDEO),
    '-map', '0:v:0', '-vf', "fps=2,scale='min(1600,iw)':-2",
    '-q:v', '2', str(IMAGES / 'frame-%06d.jpg')
], check=True)
frames = sorted(IMAGES.glob('*.jpg'))
print('Frames extracted:', len(frames))
assert len(frames) >= 10, 'Too few frames. Check the selected video.'


## 5. Estimate camera poses with the Travel Universe COLMAP adapter
This step may fail if the video does not contain enough stable background detail or camera movement. A failed camera-pose solve is a valid test result; do not skip it and continue as if reconstruction succeeded.

In [ ]:
import os, sys, subprocess
# Colab has no desktop display; use Qt's offscreen platform.
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
REPO = Path('/content/Travel-universe')
if not REPO.is_dir():
    raise RuntimeError('Travel Universe source folder is missing. Please rerun Step 2 before Step 5.')
# Refresh the cloned source so this runtime gets the latest committed worker fix.
subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
WORKER_DIR = REPO / 'apps' / 'worker'
PIPELINE_FILE = WORKER_DIR / 'colmap_pipeline.py'
print('Worker folder:', WORKER_DIR, 'exists:', WORKER_DIR.is_dir())
print('COLMAP adapter:', PIPELINE_FILE, 'exists:', PIPELINE_FILE.is_file())
if not PIPELINE_FILE.is_file():
    raise FileNotFoundError(f'Expected COLMAP adapter was not found: {PIPELINE_FILE}')
sys.path.insert(0, str(WORKER_DIR))
# If Step 5 is retried in the same Colab kernel, discard any older imported copy.
sys.modules.pop('colmap_pipeline', None)
from colmap_pipeline import run_colmap_sparse_reconstruction
colmap_result = run_colmap_sparse_reconstruction(image_dir=IMAGES, output_dir=DATA)
print(colmap_result)
MODEL = Path(colmap_result['sparse_model_directory'])
required = ['cameras.bin', 'images.bin', 'points3D.bin']
assert all((MODEL / name).is_file() and (MODEL / name).stat().st_size > 0 for name in required)
print('COLMAP model validated:', MODEL)


## 6. Train Gaussian Splats
This uses the official gsplat COLMAP example trainer. The first run is a 7,000-step test, not a promise of production quality. It can take time and may exceed free GPU availability.

In [ ]:
RESULTS = Path('/content/tu_results')
RESULTS.mkdir(parents=True, exist_ok=True)
train_command = [
    sys.executable, '/content/gsplat/examples/simple_trainer.py', 'default',
    '--data_dir', str(DATA), '--data_factor', '2', '--result_dir', str(RESULTS),
    '--max_steps', '7000', '--save_ply', '--ply_steps', '7000',
    '--save_steps', '7000', '--eval_steps', '7000', '--disable_viewer'
]
print('Starting gsplat training. Keep this Colab tab open while it runs.')
subprocess.run(train_command, cwd='/content/gsplat', check=True)
ply_files = sorted((RESULTS / 'ply').rglob('*.ply'), key=lambda p: p.stat().st_mtime)
print('PLY files found:', [str(p) for p in ply_files])
assert ply_files and ply_files[-1].stat().st_size > 0, 'Training did not produce a non-empty PLY file.'
PLY = ply_files[-1]
print('Selected PLY:', PLY, 'bytes:', PLY.stat().st_size)


## 7. Convert PLY to the SPZ format used by the Travel Universe viewer
This is a real format conversion, not a file rename. The viewer uses the Spark Gaussian Splat loader.

In [ ]:
import subprocess
SPZ = Path('/content/TravelUniverse_scene.spz')
print('Converting trained Gaussian PLY to SPZ using the official SPZ converter...')
subprocess.run([str(SPZ_CONVERTER), str(PLY), str(SPZ)], check=True)
assert SPZ.is_file() and SPZ.stat().st_size > 32, 'SPZ export failed or produced an empty file.'
with SPZ.open('rb') as f:
    magic = f.read(4)
assert magic == b'NGSP', f'Unexpected SPZ file header: {magic!r}'
print('SPZ created and header validated:', SPZ.name, 'bytes:', SPZ.stat().st_size)


## 8. Download the 3D scene to your iPad
Download this SPZ file to your iPad Files app. Do not upload it to the public GitHub repository. We will use it in the next app integration step.

In [ ]:
from google.colab import files
files.download(str(SPZ))


## Completion checklist
- COLMAP produced a valid sparse model (`cameras.bin`, `images.bin`, `points3D.bin`).
- gsplat training completed and produced a non-empty PLY.
- SPZ conversion produced a non-empty file containing Gaussian points.
- The downloaded SPZ still needs to be opened in the Travel Universe viewer and visually checked before the app can call the world ready.

**Important:** this notebook is a manually started cloud processing tool. It does not create a persistent API or automatically connect the public website's upload form to Colab. Free Colab GPU availability and session lifetime are controlled by Google. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html).